# Inventory Data Cleaning

## Objective

Clean and validate warehouse inventory data before loading into the analytics database.

## Cleaning Tasks

- Identify missing inventory attributes
- Validate inventory quantities
- Check reorder levels
- Identify duplicate inventory records
- Validate warehouse/product relationships
- Export cleaned inventory data

In [1]:
import pandas as pd
from pathlib import Path

In [3]:
raw_path = Path("../data/raw/inventory.csv")

clean_path = Path("../data/cleaned/inventory_clean.csv")

In [4]:
inventory = pd.read_csv(raw_path)

inventory.head()

,inventory_id,warehouse,product_id,inventory_cases,reorder_level,last_updated
0,1,Los Angeles DC,1001,3143,128.0,2025-07-01
1,2,Los Angeles DC,1002,282,960.0,2025-07-01
2,3,Los Angeles DC,1003,2700,467.0,2025-07-01
3,4,Los Angeles DC,1004,3312,614.0,2025-07-01
4,5,Los Angeles DC,1005,730,138.0,2025-07-01


In [5]:
inventory.shape

(170, 6)

In [6]:
inventory.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 170 entries, 0 to 169
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   inventory_id     170 non-null    int64  
 1   warehouse        170 non-null    object 
 2   product_id       170 non-null    int64  
 3   inventory_cases  170 non-null    int64  
 4   reorder_level    169 non-null    float64
 5   last_updated     170 non-null    object 
dtypes: float64(1), int64(3), object(2)
memory usage: 8.1+ KB


In [7]:
inventory.isnull().sum()

inventory_id       0
warehouse          0
product_id         0
inventory_cases    0
reorder_level      1
last_updated       0
dtype: int64

In [8]:
inventory_clean = inventory.copy()

In [9]:
inventory_clean.duplicated(
    subset=["warehouse","product_id"]
).sum()

5

In [10]:
inventory_clean[
    inventory_clean["inventory_cases"] < 0
]

,inventory_id,warehouse,product_id,inventory_cases,reorder_level,last_updated
10,11,Los Angeles DC,1011,-250,568.0,2025-07-01


In [11]:
inventory_clean[
    inventory_clean["reorder_level"].isna()
]

,inventory_id,warehouse,product_id,inventory_cases,reorder_level,last_updated
50,51,Dallas DC,1017,4436,NaN,2025-07-01


In [12]:
inventory_clean[
    inventory_clean["inventory_cases"] 
    < inventory_clean["reorder_level"]
]

,inventory_id,warehouse,product_id,inventory_cases,reorder_level,last_updated
1,2,Los Angeles DC,1002,282,960.0,2025-07-01
10,11,Los Angeles DC,1011,-250,568.0,2025-07-01
15,16,Los Angeles DC,1016,575,662.0,2025-07-01
30,31,Los Angeles DC,1031,137,368.0,2025-07-01
54,55,Dallas DC,1020,371,651.0,2025-07-01
65,66,Dallas DC,1032,128,872.0,2025-07-01
67,68,Dallas DC,1034,201,352.0,2025-07-01
95,96,Chicago DC,1028,500,570.0,2025-07-01
100,101,Chicago DC,1033,500,1000.0,2025-07-01
104,105,Atlanta DC,1003,369,397.0,2025-07-01


In [14]:
inventory_clean[
    inventory_clean.duplicated(
        subset=["warehouse","product_id"],
        keep=False
    )
].sort_values(
    ["warehouse","product_id"]
)

,inventory_id,warehouse,product_id,inventory_cases,reorder_level,last_updated
121,122,Atlanta DC,1020,3553,110.0,2025-07-01
122,123,Atlanta DC,1020,3237,800.0,2025-07-01
87,88,Chicago DC,1020,1669,886.0,2025-07-01
88,89,Chicago DC,1020,3602,704.0,2025-07-01
53,54,Dallas DC,1020,3607,684.0,2025-07-01
54,55,Dallas DC,1020,371,651.0,2025-07-01
19,20,Los Angeles DC,1020,1257,163.0,2025-07-01
20,21,Los Angeles DC,1020,3982,297.0,2025-07-01
155,156,Phoenix DC,1020,1009,348.0,2025-07-01
156,157,Phoenix DC,1020,1668,714.0,2025-07-01


In [15]:
inventory_clean.loc[
    inventory_clean["inventory_cases"] < 0,
    "inventory_cases"
] = 0

In [16]:
inventory_clean["reorder_level"].median()

595.0

In [17]:
inventory_clean["reorder_level"] = (
    inventory_clean["reorder_level"]
    .fillna(
        inventory_clean["reorder_level"].median()
    )
)

In [19]:
inventory_clean["reorder_level"] = (
    inventory_clean["reorder_level"]
    .fillna(
        inventory_clean["reorder_level"].median()
    )
)

In [20]:
inventory_clean = (
    inventory_clean
    .groupby(
        ["warehouse", "product_id"],
        as_index=False
    )
    .agg({
        "inventory_id": "first",
        "inventory_cases": "sum",
        "reorder_level": "mean",
        "last_updated": "max"
    })
)

In [21]:
inventory_clean.duplicated(
    subset=["warehouse","product_id"]
).sum()

0

In [22]:
inventory_clean.shape

(165, 6)

Inventory Data Quality Findings:

- Identified duplicate warehouse/product inventory records.
- Consolidated duplicate inventory balances by warehouse and product.
- Corrected negative inventory values.
- Filled missing reorder thresholds.

In [24]:
inventory_clean.isnull().sum()

warehouse          0
product_id         0
inventory_id       0
inventory_cases    0
reorder_level      0
last_updated       0
dtype: int64

In [25]:
inventory_clean.duplicated(
    subset=["warehouse","product_id"]
).sum()

0

In [26]:
(inventory_clean["inventory_cases"] < 0).sum()

0

In [27]:
inventory_clean.shape

(165, 6)

In [28]:
clean_path = Path("../data/cleaned/inventory_clean.csv")

inventory_clean.to_csv(
    clean_path,
    index=False
)

In [2]:
import pandas as pd

inventory_clean = pd.read_csv(
    "../data/cleaned/inventory_clean.csv"
)

In [3]:
inventory_clean.dtypes

warehouse           object
product_id           int64
inventory_id         int64
inventory_cases      int64
reorder_level      float64
last_updated        object
dtype: object

In [4]:
inventory_clean["reorder_level"] = (
    inventory_clean["reorder_level"]
    .astype(int)
)

In [5]:
inventory_clean.dtypes

warehouse          object
product_id          int64
inventory_id        int64
inventory_cases     int64
reorder_level       int64
last_updated       object
dtype: object

In [6]:
inventory_clean.to_csv(
    "../data/cleaned/inventory_clean.csv",
    index=False
)